## **Simpe Website Builder**

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool

from dotenv import load_dotenv

from pathlib import Path
import os

load_dotenv()

# ============================================================
# ENVIRONMENT
# ============================================================

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are Website Builder Agent, an expert full-stack software engineer.

Your job is to build, modify, debug, and maintain web applications.

You have access to tools for:

- Creating files
- Reading files
- Editing files
- Deleting files
- Listing project files
- Searching code
- Creating directories

WORKFLOW:

1. Always inspect project structure first.
2. Read relevant files before editing.
3. Understand architecture before making changes.
4. Preserve existing functionality.
5. Use best practices.
6. Build production-ready code.
7. Use responsive design.
8. Use semantic HTML.
9. Create reusable components.
10. Verify consistency after changes.

RULES:

- Never overwrite files without checking.
- Never delete files unless explicitly requested.
- Always explain modifications made.
- Think like a professional software engineer.
"""


# ============================================================
# TOOLS
# ============================================================

@tool
def create_directory(path: str) -> str:
    """
    Create a directory.

    Args:
        path: Directory path.

    Returns:
        Success or error message.
    """
    try:
        Path(path).mkdir(parents=True, exist_ok=True)
        return f"Directory created: {path}"

    except Exception as e:
        return f"Error creating directory: {str(e)}"


@tool
def create_file(filepath: str, content: str) -> str:
    """
    Create a new file.

    Use when a new page, component,
    stylesheet, API route, or configuration
    file is required.

    Args:
        filepath: Relative file path.
        content: Complete file content.

    Returns:
        Success or failure message.
    """

    try:
        path = Path(filepath)

        if path.exists():
            return f"Error: File already exists -> {filepath}"

        path.parent.mkdir(parents=True, exist_ok=True)

        with open(path, "w", encoding="utf-8") as file:
            file.write(content)

        return f"Successfully created file: {filepath}"

    except Exception as e:
        return f"Error creating file: {str(e)}"


@tool
def read_file(filepath: str) -> str:
    """
    Read the contents of a file.

    Use before modifying existing code.

    Args:
        filepath: Path to file.

    Returns:
        Complete file content.
    """

    try:
        with open(filepath, "r", encoding="utf-8") as file:
            return file.read()

    except Exception as e:
        return f"Error reading file: {str(e)}"


@tool
def read_multiple_files(filepaths: list[str]) -> str:
    """
    Read multiple files simultaneously.

    Useful when understanding features
    spread across several files.

    Args:
        filepaths: List of file paths.

    Returns:
        Contents of all files.
    """

    output = []

    for filepath in filepaths:

        try:

            with open(filepath, "r", encoding="utf-8") as file:

                output.append(
                    f"\n\n===== {filepath} =====\n"
                    + file.read()
                )

        except Exception as e:

            output.append(
                f"\n\n===== {filepath} =====\n"
                f"ERROR: {str(e)}"
            )

    return "".join(output)


@tool
def edit_file(filepath: str, new_content: str) -> str:
    """
    Modify an existing file.

    Always read the file before editing.

    Args:
        filepath: File path.
        new_content: Updated content.

    Returns:
        Success or failure message.
    """

    try:

        if not os.path.exists(filepath):
            return f"Error: File does not exist -> {filepath}"

        with open(filepath, "w", encoding="utf-8") as file:
            file.write(new_content)

        return f"Successfully updated file: {filepath}"

    except Exception as e:
        return f"Error editing file: {str(e)}"


@tool
def delete_file(filepath: str) -> str:
    """
    Delete a file.

    Use only when explicitly requested.

    Args:
        filepath: File path.

    Returns:
        Success or failure message.
    """

    try:

        if not os.path.exists(filepath):
            return f"Error: File does not exist -> {filepath}"

        os.remove(filepath)

        return f"Successfully deleted file: {filepath}"

    except Exception as e:
        return f"Error deleting file: {str(e)}"


@tool
def list_files(directory: str = ".") -> str:
    """
    List project files recursively.

    Useful for understanding project structure.

    Args:
        directory: Root directory.

    Returns:
        Tree of files and folders.
    """

    try:

        result = []

        for root, dirs, files in os.walk(directory):

            level = root.replace(directory, "").count(os.sep)

            indent = " " * 4 * level

            result.append(f"{indent}{os.path.basename(root)}/")

            subindent = " " * 4 * (level + 1)

            for file in files:
                result.append(f"{subindent}{file}")

        return "\n".join(result)

    except Exception as e:
        return f"Error listing files: {str(e)}"


@tool
def search_code(query: str, directory: str = ".") -> str:
    """
    Search project files for specific text.

    Useful for finding:
    - Components
    - Routes
    - APIs
    - Functions
    - Imports

    Args:
        query: Search term.
        directory: Directory to search.

    Returns:
        Matching results.
    """

    matches = []

    try:

        for root, _, files in os.walk(directory):

            for file in files:

                filepath = os.path.join(root, file)

                try:

                    with open(
                        filepath,
                        "r",
                        encoding="utf-8",
                        errors="ignore"
                    ) as f:

                        for line_number, line in enumerate(f, start=1):

                            if query.lower() in line.lower():

                                matches.append(
                                    f"{filepath}"
                                    f" [Line {line_number}] "
                                    f"{line.strip()}"
                                )

                except:
                    pass

        if not matches:
            return f"No matches found for '{query}'."

        return "\n".join(matches)

    except Exception as e:
        return f"Error searching code: {str(e)}"


# ============================================================
# AGENT
# ============================================================

agent = create_agent(
    name="Website Builder Agent",

    model=ChatGroq(
        model="qwen/qwen3-32b"
    ),

    system_prompt=SYSTEM_PROMPT,

    tools=[
        create_directory,
        create_file,
        read_file,
        read_multiple_files,
        edit_file,
        delete_file,
        list_files,
        search_code,
    ]
)


# ============================================================
# DEMO
# ============================================================

for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content":
                "Create a modern landing page for FIFA World Cup 2026 using HTML,CSS and JS."
            }
        ]
    },
    stream_mode="messages"
):

    token, metadata = chunk

    if hasattr(token, "content") and token.content:
        print(token.content, end="", flush=True)